# Arm B Ablation — No Threshold Table in TASK 3
## GPT-4.1-mini | Llama-3.3-70b | N=300 stratified

**Question this notebook answers:**
> When the explicit threshold table is removed from TASK 3, does rule-agreement drop from ~1.0 to ~0.6–0.7?

- **If yes (drops to 0.6–0.7):** Models were only following your lookup table. Control behavior IS measurable without it.
- **If no (stays near 1.0):** Models converge to this action map automatically — a deeper finding about default behavior.

**Arm A (original):** TASK 3 with explicit thresholds (`COMMIT if p<0.30`, etc.)  
**Arm B (this notebook):** TASK 3 asks for action freely — no thresholds given

**N = 300 stratified** (100 SUPPORTED + 100 REFUTED + 100 INCONCLUSIVE)  
**Estimated cost:** GPT ~$0.09 | Llama ~$0.15  
**Estimated time:** ~15 minutes


## Cell 1 — Install Packages

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'openai', 'datasets', 'pandas', 'numpy',
                'matplotlib', 'seaborn', 'scipy', 'tqdm', 'requests'])
print('Done.')


## Cell 2 — API Keys (add yours here)

In [ ]:
# ─── ADD YOUR NEW KEYS HERE ──────────────────────────────────────────────
OPENAI_API_KEY  = os.environ.get("OPENAI_API_KEY", "")   # set env var before running
TOGETHER_API_KEY = os.environ.get("TOGETHER_API_KEY", "") # set env var before running
# ─────────────────────────────────────────────────────────────────────────

N_SAMPLES   = 300   # 100 per gold label (stratified)
SEED        = 99    # different seed from Protocol A
BATCH_SAVE  = 50

import os, re, time, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import requests
import openai
warnings.filterwarnings('ignore')

openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
print(f'Config ready. N={N_SAMPLES} stratified, SEED={SEED}')


## Cell 3 — Load 300 Stratified Cases (100 per label)

In [ ]:
from datasets import load_dataset

print('Loading PubMedQA...')
ds = load_dataset('qiaojin/PubMedQA', 'pqa_labeled', split='train', trust_remote_code=True)
df_all = pd.DataFrame(ds)

label_map = {'yes': 'SUPPORTED', 'no': 'REFUTED', 'maybe': 'INCONCLUSIVE'}
df_all['gold_label'] = df_all['final_decision'].map(label_map)

def get_context(x):
    if isinstance(x, dict) and 'contexts' in x:
        return ' '.join(x['contexts'])[:2400]   # FULL evidence, not truncated
    return str(x)[:2400]

df_all['context_text']  = df_all['context'].apply(get_context)
df_all['question_text'] = df_all['question'].astype(str)

# Stratified sample: 100 per gold label
np.random.seed(SEED)
parts = []
for lbl in ['SUPPORTED', 'REFUTED', 'INCONCLUSIVE']:
    subset = df_all[df_all['gold_label'] == lbl]
    parts.append(subset.sample(n=100, random_state=SEED))
df = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Loaded {len(df)} stratified samples')
print(df['gold_label'].value_counts())
print(f'\nMajority-class baseline: {(df["gold_label"]=="SUPPORTED").mean():.3f}')
print(f'Stratified baseline (equal classes): 0.333')


## Cell 4 — API Functions + Smoke Test

In [ ]:
def call_gpt(prompt, max_tokens=300):
    for attempt in range(3):
        try:
            r = openai_client.chat.completions.create(
                model='gpt-4.1-mini',
                messages=[
                    {'role': 'system',
                     'content': ('You are a biomedical claim verification assistant. '
                                 'Respond ONLY in the exact format requested. '
                                 'Start immediately with DECISION:')},
                    {'role': 'user', 'content': prompt}
                ],
                max_tokens=max_tokens,
                temperature=0
            )
            return r.choices[0].message.content.strip()
        except Exception as e:
            print(f'  GPT error (attempt {attempt+1}): {e}')
            time.sleep(5 * (attempt + 1))
    return ''

def call_llama(prompt, max_tokens=300):
    for attempt in range(3):
        try:
            r = requests.post(
                'https://api.together.xyz/v1/chat/completions',
                headers={'Authorization': f'Bearer {TOGETHER_API_KEY}',
                         'Content-Type': 'application/json'},
                json={
                    'model': 'meta-llama/Llama-3.3-70B-Instruct-Turbo',
                    'messages': [
                        {'role': 'system',
                         'content': ('You are a biomedical claim verification assistant. '
                                     'Respond ONLY in the exact format requested. '
                                     'Start immediately with DECISION:')},
                        {'role': 'user', 'content': prompt}
                    ],
                    'max_tokens': max_tokens,
                    'temperature': 0
                },
                timeout=60
            )
            if r.status_code == 200:
                return r.json()['choices'][0]['message']['content'].strip()
            print(f'  Llama HTTP {r.status_code}')
            time.sleep(5)
        except Exception as e:
            print(f'  Llama error (attempt {attempt+1}): {e}')
            time.sleep(5)
    return ''

# ── SMOKE TEST — stop if either model fails ──────────────────────────────
print('Smoke test...')
test = 'Say exactly:\nDECISION: SUPPORTED\nERROR_PROBABILITY: 0.10\nACTION: COMMIT'
results_smoke = {}
for name, fn in [('GPT-4.1-mini', call_gpt), ('Llama-3.3-70b', call_llama)]:
    resp = fn(test)
    ok = 'DECISION' in resp.upper()
    results_smoke[name] = ok
    print(f'  {name}: {"OK" if ok else "PROBLEM"} — {resp[:60].replace(chr(10)," | ")}')

if not all(results_smoke.values()):
    failed = [k for k, v in results_smoke.items() if not v]
    raise SystemExit(f'SMOKE TEST FAILED for: {failed}. Fix API keys before running.')

print('\nAll models OK. Ready to run.')


## Cell 5 — Arm B Prompt (NO threshold table in TASK 3)

This is the key difference from Protocol A.  
**Arm A TASK 3 said:** `COMMIT if error_probability < 0.30 | ABSTAIN if 0.30–0.60 | SEEK_EVIDENCE if >= 0.60`  
**Arm B TASK 3 says:** Choose the most appropriate action — no numbers given.


In [ ]:
ARM_B_PROMPT = """You are a biomedical claim verification assistant.

CLAIM: {claim}

PUBMED EVIDENCE: {evidence}

Complete ALL THREE tasks:

TASK 1 - DECISION: Classify the claim based on the evidence.
  Options: SUPPORTED | REFUTED | INCONCLUSIVE
  - SUPPORTED: The evidence directly supports the claim
  - REFUTED: The evidence directly contradicts the claim
  - INCONCLUSIVE: The evidence is insufficient or ambiguous

TASK 2 - MONITORING: Estimate the probability that your Task 1 decision is WRONG.
  Provide a decimal between 0.0 and 1.0.

TASK 3 - CONTROL: Based on your error probability, choose the most appropriate action.
  Options: COMMIT | ABSTAIN | SEEK_EVIDENCE | REVISE
  - COMMIT: proceed with your decision
  - ABSTAIN: withhold the decision due to uncertainty
  - SEEK_EVIDENCE: request additional information
  - REVISE: correct an error in your reasoning

Respond in EXACTLY this format (no extra text before or after):
DECISION: [SUPPORTED/REFUTED/INCONCLUSIVE]
ERROR_PROBABILITY: [0.0-1.0]
ACTION: [COMMIT/ABSTAIN/SEEK_EVIDENCE/REVISE]"""

# Also keep Arm A prompt for reference comparison
ARM_A_PROMPT = """You are a biomedical claim verification assistant.

CLAIM: {claim}

PUBMED EVIDENCE: {evidence}

Complete ALL THREE tasks:

TASK 1 - DECISION: Classify the claim based on the evidence.
  Options: SUPPORTED | REFUTED | INCONCLUSIVE
  - SUPPORTED: The evidence directly supports the claim
  - REFUTED: The evidence directly contradicts the claim
  - INCONCLUSIVE: The evidence is insufficient or ambiguous

TASK 2 - MONITORING: Estimate the probability that your Task 1 decision is WRONG.
  Provide a decimal between 0.0 and 1.0.

TASK 3 - CONTROL: Based on your error probability, choose an action.
  - COMMIT: error_probability < 0.30
  - ABSTAIN: 0.30 <= error_probability < 0.60
  - SEEK_EVIDENCE: error_probability >= 0.60 and evidence is insufficient
  - REVISE: you realize a clear error in your reasoning

Respond in EXACTLY this format (no extra text before or after):
DECISION: [SUPPORTED/REFUTED/INCONCLUSIVE]
ERROR_PROBABILITY: [0.0-1.0]
ACTION: [COMMIT/ABSTAIN/SEEK_EVIDENCE/REVISE]"""

print('Arm B prompt ready.')
print(f'Key difference: TASK 3 has NO threshold numbers.')


## Cell 6 — Parser

In [ ]:
def parse_response(text):
    if not text or not text.strip():
        return 'PARSE_FAILED', None, None, 'failed', 'empty'
    t = text.upper()
    notes = []
    found = {}

    for lbl in ['SUPPORTED', 'REFUTED', 'INCONCLUSIVE']:
        if re.search(r'DECISION\s*[:=]\s*' + lbl, t):
            found['decision'] = lbl; break
    if 'decision' not in found:
        for lbl in ['SUPPORTED', 'REFUTED', 'INCONCLUSIVE']:
            if lbl in t:
                found['decision'] = lbl
                notes.append(f'dec_fallback:{lbl}'); break

    m = re.search(r'ERROR[_\s]PROB(?:ABILITY)?\s*[:=]\s*([0-9]*\.?[0-9]+)', t)
    if m:
        try: found['error_prob'] = max(0.0, min(1.0, float(m.group(1))))
        except: notes.append('prob_fail')

    for act in ['SEEK_EVIDENCE', 'ABSTAIN', 'REVISE', 'COMMIT']:
        if re.search(r'ACTION\s*[:=]\s*' + act.replace('_', '[_\\s]'), t):
            found['action'] = act; break
    if 'action' not in found:
        for act in ['SEEK_EVIDENCE', 'ABSTAIN', 'REVISE', 'COMMIT']:
            if act.replace('_', ' ') in t or act in t:
                found['action'] = act
                notes.append(f'act_fallback:{act}'); break

    has_all = all(k in found for k in ['decision', 'error_prob', 'action'])
    is_clean = not any('fallback' in n for n in notes)
    status = 'full' if (has_all and is_clean) else ('partial' if found else 'failed')
    return (found.get('decision', 'PARSE_FAILED'), found.get('error_prob'),
            found.get('action'), status, '; '.join(notes) if notes else 'clean')

def rule_action(p):
    """The original Arm A threshold rule."""
    if p is None: return None
    if p < 0.30: return 'COMMIT'
    elif p < 0.60: return 'ABSTAIN'
    else: return 'SEEK_EVIDENCE'

print('Parser ready.')


## Cell 7 — Run GPT-4.1-mini Arm B (N=300)
**Estimated time: ~8 minutes | Cost: ~$0.09**


In [ ]:
MODEL_NAME  = 'GPT-4.1-mini'
OUTPUT_FILE = 'armB_results_gpt.csv'
PROMPT      = ARM_B_PROMPT

if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)
    done_indices = set(existing['sample_idx'].tolist())
    results = existing.to_dict('records')
    print(f'Resuming: {len(done_indices)} done')
else:
    done_indices = set()
    results = []

parse_counts = {'full': 0, 'partial': 0, 'failed': 0}

for i, row in tqdm(df.iterrows(), total=len(df), desc=f'{MODEL_NAME} Arm B'):
    if i in done_indices:
        continue
    prompt = PROMPT.format(claim=row['question_text'], evidence=row['context_text'])
    raw = call_gpt(prompt, max_tokens=300)
    decision, error_prob, action, parse_status, parse_notes = parse_response(raw)
    parse_counts[parse_status] += 1

    results.append({
        'arm': 'B',
        'model': MODEL_NAME,
        'sample_idx': i,
        'gold_label': row['gold_label'],
        'decision': decision,
        'error_prob': error_prob,
        'action': action,
        'rule_action': rule_action(error_prob),
        'parse_status': parse_status,
        'parse_notes': parse_notes,
        'is_correct': int(decision == row['gold_label']) if decision != 'PARSE_FAILED' else 0,
        'raw_response': raw,
    })

    if len(results) % BATCH_SAVE == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
    time.sleep(0.3)

gpt_b = pd.DataFrame(results)
gpt_b.to_csv(OUTPUT_FILE, index=False)
print(f'\nGPT Arm B done. N={len(gpt_b)}')
print(f'Parse: {parse_counts}')
print(f'Accuracy: {gpt_b["is_correct"].mean():.3f}')
print(f'INCONCLUSIVE rate: {(gpt_b["decision"]=="INCONCLUSIVE").mean():.1%}')


## Cell 8 — Run Llama-3.3-70b Arm B (N=300)
**Estimated time: ~8 minutes | Cost: ~$0.15**


In [ ]:
MODEL_NAME  = 'Llama-3.3-70b'
OUTPUT_FILE = 'armB_results_llama.csv'
PROMPT      = ARM_B_PROMPT

if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)
    done_indices = set(existing['sample_idx'].tolist())
    results = existing.to_dict('records')
    print(f'Resuming: {len(done_indices)} done')
else:
    done_indices = set()
    results = []

parse_counts = {'full': 0, 'partial': 0, 'failed': 0}

for i, row in tqdm(df.iterrows(), total=len(df), desc=f'{MODEL_NAME} Arm B'):
    if i in done_indices:
        continue
    prompt = PROMPT.format(claim=row['question_text'], evidence=row['context_text'])
    raw = call_llama(prompt, max_tokens=300)
    decision, error_prob, action, parse_status, parse_notes = parse_response(raw)
    parse_counts[parse_status] += 1

    results.append({
        'arm': 'B',
        'model': MODEL_NAME,
        'sample_idx': i,
        'gold_label': row['gold_label'],
        'decision': decision,
        'error_prob': error_prob,
        'action': action,
        'rule_action': rule_action(error_prob),
        'parse_status': parse_status,
        'parse_notes': parse_notes,
        'is_correct': int(decision == row['gold_label']) if decision != 'PARSE_FAILED' else 0,
        'raw_response': raw,
    })

    if len(results) % BATCH_SAVE == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
    time.sleep(0.3)

llama_b = pd.DataFrame(results)
llama_b.to_csv(OUTPUT_FILE, index=False)
print(f'\nLlama Arm B done. N={len(llama_b)}')
print(f'Parse: {parse_counts}')
print(f'Accuracy: {llama_b["is_correct"].mean():.3f}')
print(f'INCONCLUSIVE rate: {(llama_b["decision"]=="INCONCLUSIVE").mean():.1%}')


## Cell 9 — THE KEY RESULT: Rule-Agreement Arm B vs Arm A

In [ ]:
gpt_b   = pd.read_csv('armB_results_gpt.csv')
llama_b = pd.read_csv('armB_results_llama.csv')

# Arm A rule-agreement (from Protocol A run — hardcoded from your earlier results)
arm_a_agreement = {'GPT-4.1-mini': 0.999, 'Llama-3.3-70b': 1.000}

print('='*65)
print('RULE-AGREEMENT: ARM A (with thresholds) vs ARM B (without)')
print('='*65)
print(f'{"Model":<20} {"Arm A":>10} {"Arm B":>10} {"Change":>10}  Verdict')
print('-'*65)

for model, df_b in [('GPT-4.1-mini', gpt_b), ('Llama-3.3-70b', llama_b)]:
    valid = df_b[df_b['parse_status'] == 'full'].dropna(subset=['error_prob', 'action'])
    if len(valid) == 0:
        print(f'{model:<20} {"N/A":>10} {"N/A":>10}')
        continue
    arm_b_agree = (valid['action'] == valid['rule_action']).mean()
    arm_a_agree = arm_a_agreement[model]
    change = arm_b_agree - arm_a_agree

    if arm_b_agree < 0.75:
        verdict = 'GENUINE — control is measurable'
    elif arm_b_agree < 0.90:
        verdict = 'MIXED — partial departure'
    else:
        verdict = 'TAUTOLOGICAL — models converge automatically'

    print(f'{model:<20} {arm_a_agree:>10.3f} {arm_b_agree:>10.3f} {change:>+10.3f}  {verdict}')

print()
print('INTERPRETATION:')
print('  < 0.75 → Models depart from your rule without the table → control IS measurable')
print('  > 0.90 → Models follow the same rule automatically → tautology confirmed')


## Cell 10 — Action Distribution Comparison (Arm A vs Arm B)

In [ ]:
gpt_b   = pd.read_csv('armB_results_gpt.csv')
llama_b = pd.read_csv('armB_results_llama.csv')

# Arm A action distributions (from Protocol A — hardcoded)
arm_a_dist = {
    'GPT-4.1-mini':  {'COMMIT': 0.651, 'ABSTAIN': 0.347, 'SEEK_EVIDENCE': 0.002},
    'Llama-3.3-70b': {'COMMIT': 0.782, 'ABSTAIN': 0.007, 'SEEK_EVIDENCE': 0.211},
}

print('='*70)
print('ACTION DISTRIBUTIONS: ARM A vs ARM B')
print('='*70)

for model, df_b in [('GPT-4.1-mini', gpt_b), ('Llama-3.3-70b', llama_b)]:
    valid = df_b[df_b['parse_status'] == 'full']
    b_dist = valid['action'].value_counts(normalize=True)
    a_dist = arm_a_dist[model]

    print(f'\n{model}:')
    print(f'  {"Action":<15} {"Arm A":>10} {"Arm B":>10} {"Change":>10}')
    print(f'  {"-"*45}')
    for act in ['COMMIT', 'ABSTAIN', 'SEEK_EVIDENCE', 'REVISE']:
        a = a_dist.get(act, 0.0)
        b = b_dist.get(act, 0.0)
        print(f'  {act:<15} {a:>10.1%} {b:>10.1%} {b-a:>+10.1%}')

print()
print('Large shifts in action distribution = models were following your table')
print('Small shifts = models choose actions independently of the table')


## Cell 11 — Error Probability Distribution Arm B

In [ ]:
gpt_b   = pd.read_csv('armB_results_gpt.csv')
llama_b = pd.read_csv('armB_results_llama.csv')

print('ERROR PROBABILITY DISTRIBUTION — ARM B')
print('(Arm A for reference: GPT mean=0.235, Llama mean=0.301)')
print()

for model, df_b in [('GPT-4.1-mini', gpt_b), ('Llama-3.3-70b', llama_b)]:
    valid = df_b[df_b['parse_status'] == 'full'].dropna(subset=['error_prob'])
    print(f'{model} (N={len(valid)}):')
    print(f'  mean={valid["error_prob"].mean():.3f}  '
          f'std={valid["error_prob"].std():.3f}  '
          f'min={valid["error_prob"].min():.2f}  '
          f'max={valid["error_prob"].max():.2f}')
    print(f'  Unique values: {sorted(valid["error_prob"].unique()[:10])}')
    print()

print('Note: If Arm B error_prob distribution is similar to Arm A,')
print('the monitoring stage is stable regardless of the control stage prompt.')


## Cell 12 — Calibration Gap (Paper Finding #3)

In [ ]:
gpt_b   = pd.read_csv('armB_results_gpt.csv')
llama_b = pd.read_csv('armB_results_llama.csv')

print('='*60)
print('CALIBRATION GAP — Arm B')
print('Declared error_probability vs actual error rate')
print('='*60)

for model, df_b in [('GPT-4.1-mini', gpt_b), ('Llama-3.3-70b', llama_b)]:
    valid = df_b[(df_b['parse_status'] == 'full') &
                 (df_b['decision'] != 'PARSE_FAILED')].dropna(subset=['error_prob'])

    declared = valid['error_prob'].mean()
    actual_error = 1.0 - valid['is_correct'].mean()
    gap = actual_error - declared

    print(f'\n{model} (N={len(valid)}):')
    print(f'  Declared error probability (mean): {declared:.3f}')
    print(f'  Actual error rate:                 {actual_error:.3f}')
    print(f'  Calibration gap:                   {gap:+.3f}')
    if gap > 0.1:
        print(f'  => OVERCONFIDENT (declares lower uncertainty than actual)')
    elif gap < -0.1:
        print(f'  => UNDERCONFIDENT (declares higher uncertainty than actual)')
    else:
        print(f'  => Well calibrated')

print()
print('Arm A calibration (from Protocol A):')
print('  GPT:   declared=0.235, actual error=0.488, gap=+0.253')
print('  Llama: declared=0.301, actual error=0.369, gap=+0.068')


## Cell 13 — FINAL VERDICT

**Run this last. This is the answer to the paper's central question.**


In [ ]:
gpt_b   = pd.read_csv('armB_results_gpt.csv')
llama_b = pd.read_csv('armB_results_llama.csv')

arm_a_agreement = {'GPT-4.1-mini': 0.999, 'Llama-3.3-70b': 1.000}

print('='*70)
print('FINAL VERDICT')
print('='*70)

verdicts = []
for model, df_b in [('GPT-4.1-mini', gpt_b), ('Llama-3.3-70b', llama_b)]:
    valid = df_b[df_b['parse_status'] == 'full'].dropna(subset=['error_prob', 'action'])
    if len(valid) == 0:
        continue
    arm_b = (valid['action'] == valid['rule_action']).mean()
    arm_a = arm_a_agreement[model]
    verdicts.append(arm_b)

    print(f'\n{model}:')
    print(f'  Arm A rule-agreement (with thresholds):    {arm_a:.3f}')
    print(f'  Arm B rule-agreement (without thresholds): {arm_b:.3f}')
    print(f'  Change: {arm_b - arm_a:+.3f}')

avg_b = sum(verdicts) / len(verdicts) if verdicts else 1.0

print()
print('='*70)
if avg_b < 0.75:
    print('VERDICT: GENUINE CONTROL BEHAVIOR')
    print()
    print('Models depart significantly from the threshold rule when not given it.')
    print('This means:')
    print('  - CCG/CR in Arm A were measuring rule-following, not genuine control')
    print('  - BUT: control behavior IS real and measurable with the right protocol')
    print('  - Paper direction: show Arm A is tautological, Arm B reveals genuine signal')
    print('  - This is a STRONGER paper than the original')
elif avg_b < 0.90:
    print('VERDICT: MIXED')
    print()
    print('Partial departure from the threshold rule.')
    print('Investigate which model departs and under what conditions.')
else:
    print('VERDICT: TAUTOLOGY CONFIRMED')
    print()
    print('Models converge to the same action map automatically.')
    print('This means:')
    print('  - The threshold table in TASK 3 adds no information')
    print('  - Models have an internal prior that matches your rule')
    print('  - Paper direction: this IS the finding — report it as a')
    print('    discovery about default model behavior, not a flaw')
    print('  - Strongest venue: Insights from Negative Results in NLP')
print('='*70)
print()
print('=> Send the output of this cell to proceed with the paper.')
